## Changes made in this version

1. **Chat template** — prompts are now wrapped with `tokenizer.apply_chat_template(...)` instead of sent as raw text, matching how `SmolLM2-135M-Instruct` was aligned to follow instructions.
2. **More training data** — 10,000 train examples (6,500 answerable / 3,500 unanswerable) instead of 2,000.
3. **Gradient accumulation** — effective batch size 16 (`BATCH_SIZE=4 x ACCUM_STEPS=4`) for more stable gradients.
4. **More epochs / tuned LR** — `EPOCHS=6`, `LEARNING_RATE=2e-5`, 10% warmup.
5. **F1 metric** — token-overlap F1 added alongside exact match, so partially-correct answerable predictions aren't scored identically to completely wrong ones.

The two-stage "classify answerable vs. not, then extract" architecture is a bigger change (separate classifier head) and isn't included here — happy to build that as a next step if this version doesn't close the gap enough.


In [1]:
!pip install -q torch transformers datasets accelerate peft


In [2]:
from datasets import load_dataset
import json, random

SEED = 42
random.seed(SEED)
dataset = load_dataset("rajpurkar/squad_v2")

def make_example(item):
    answers = item["answers"]["text"]
    answerable = len(answers) > 0
    prompt = f"""You are a reliable QA assistant.
Use only the provided context.
If the answer is not supported by the context,
respond exactly with UNANSWERABLE.

Context: {item['context']}
Question: {item['question']}

Answer:"""
    return {
        "prompt": prompt,
        "target": answers[0] if answerable else "UNANSWERABLE",
        "answerable": answerable,
        "all_answers": list(dict.fromkeys(answers)),
        "id": item["id"],
    }

train = dataset["train"].shuffle(seed=SEED)
train_ans = train.filter(lambda x: len(x["answers"]["text"]) > 0)
train_un = train.filter(lambda x: len(x["answers"]["text"]) == 0)

# Larger training set, ~65/35 answerable/unanswerable (close to real SQuAD v2 ratio)
n_answerable = 6500
n_unanswerable = 3500
train_data = [make_example(x) for x in train_ans.select(range(n_answerable))]
train_data += [make_example(x) for x in train_un.select(range(n_unanswerable))]
random.Random(SEED).shuffle(train_data)

validation = dataset["validation"].shuffle(seed=SEED)
val_data = [make_example(x) for x in validation.select(range(300))]
test_data = [make_example(x) for x in validation.select(range(300, 800))]

for name, data in [("train", train_data), ("validation", val_data), ("test", test_data)]:
    with open(f"{name}.jsonl", "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")
    n_un = sum(not x["answerable"] for x in data)
    print(name, len(data), "unanswerable:", n_un)


README.md: 0.00B [00:00, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

Filter:   0%|          | 0/130319 [00:00<?, ? examples/s]

Filter:   0%|          | 0/130319 [00:00<?, ? examples/s]

train 10000 unanswerable: 3500
validation 300 unanswerable: 155
test 500 unanswerable: 254


In [3]:
import json
import math
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"

MAX_LENGTH = 512
BATCH_SIZE = 4
ACCUM_STEPS = 4          # effective batch size = BATCH_SIZE * ACCUM_STEPS = 16
EPOCHS = 5
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to(device)

def format_prompt(raw_prompt):
    """Wrap the instruction in the model's chat template instead of feeding
    it raw text. SmolLM2-Instruct was aligned on this format, so using it
    makes the UNANSWERABLE instruction much more reliable to follow."""
    messages = [{"role": "user", "content": raw_prompt}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

print("Model loaded.")


Device: cuda


config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded.


In [4]:
def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


train_data = read_jsonl("train.jsonl")
val_data = read_jsonl("validation.jsonl")

print("Train:", len(train_data))
print("Validation:", len(val_data))


Train: 10000
Validation: 300


In [5]:
def preprocess(example):
    prompt = format_prompt(example["prompt"])
    target = example["target"] + tokenizer.eos_token

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(target, add_special_tokens=False)["input_ids"]

    input_ids = prompt_ids + target_ids

    if len(input_ids) > MAX_LENGTH:
        return None
    label = [-100] * len(prompt_ids) + target_ids
    return {
        "input_ids": input_ids,
        "labels": label,
    }

train_example = []
for example in train_data:
    process = preprocess(example)
    if process is not None:
        train_example.append(process)

val_example = []
for example in val_data:
    process = preprocess(example)
    if process is not None:
        val_example.append(process)

print("Train:", len(train_example))
print("Validation:", len(val_example))


Train: 9923
Validation: 297


In [6]:
class QAdata_set(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = QAdata_set(train_example)
val_dataset = QAdata_set(val_example)

def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, labels, attention_mask = [], [], []

    for x in batch:
        pad_len = max_len - len(x["input_ids"])
        ids = x["input_ids"] + [tokenizer.pad_token_id] * pad_len
        lab = x["labels"] + [-100] * pad_len
        att = [1] * len(x["input_ids"]) + [0] * pad_len
        input_ids.append(ids)
        labels.append(lab)
        attention_mask.append(att)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
    }

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


In [7]:
from transformers import get_cosine_schedule_with_warmup

steps_per_epoch = math.ceil(len(train_loader) / ACCUM_STEPS)
total_steps = steps_per_epoch * EPOCHS

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)


In [8]:
def train_epoch(model, train_loader, optimizer, scheduler, device, accum_steps=ACCUM_STEPS):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, labels=labels, attention_mask=attention_mask)
        loss = outputs.loss / accum_steps
        loss.backward()
        total_loss += outputs.loss.item()

        is_last_batch = (step + 1) == len(train_loader)
        if (step + 1) % accum_steps == 0 or is_last_batch:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    return total_loss / len(train_loader)


In [9]:
@torch.no_grad
def evaluate(model, val_loader, device):
    model.eval()
    total_loss = 0
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, labels=labels, attention_mask=attention_mask)
        loss = outputs.loss
        total_loss += loss.item()

    return total_loss / len(val_loader)


In [10]:
SAVE_PATH = "./reliableqa-model/final"
best_val_loss = float("inf")

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss = evaluate(model, val_loader, device)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained(SAVE_PATH)
        tokenizer.save_pretrained(SAVE_PATH)
        print(f"  -> new best (val loss {val_loss:.4f}), saved to {SAVE_PATH}")

print(f"\nBest val loss: {best_val_loss:.4f}. Best model is saved at {SAVE_PATH}")


Epoch 1/5 | Train Loss: 1.0630 | Val Loss: 0.6774


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> new best (val loss 0.6774), saved to ./reliableqa-model/final
Epoch 2/5 | Train Loss: 0.6634 | Val Loss: 0.6323


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> new best (val loss 0.6323), saved to ./reliableqa-model/final
Epoch 3/5 | Train Loss: 0.6311 | Val Loss: 0.6270


...

  -> new best (val loss 0.6270), saved to ./reliableqa-model/final
Epoch 4/5 | Train Loss: 0.6286 | Val Loss: 0.6262


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> new best (val loss 0.6262), saved to ./reliableqa-model/final
Epoch 5/5 | Train Loss: 0.6269 | Val Loss: 0.6238


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> new best (val loss 0.6238), saved to ./reliableqa-model/final

Best val loss: 0.6238. Best model is saved at ./reliableqa-model/final


In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM

TEST_MODEL_PATH = "./reliableqa-model/final"

test_tokenizer = AutoTokenizer.from_pretrained(TEST_MODEL_PATH)
test_model = AutoModelForCausalLM.from_pretrained(TEST_MODEL_PATH)

test_model.to(device)
test_model.eval()

print("Model loaded.")

test_data = read_jsonl("test.jsonl")

print("Number of test examples:", len(test_data))


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Model loaded.
Number of test examples: 500


In [12]:
def generate_answer(model, raw_prompt, tokenizer, device):
    prompt = format_prompt(raw_prompt)
    input_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)["input_ids"].to(device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_tokens = outputs[0][input_ids.shape[1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return answer.strip()


In [13]:
example = test_data[0]
answer = generate_answer(
    test_model,
    example["prompt"],
    test_tokenizer,
    device
)
print("Question:")
print(example["prompt"])
print("\nExpected:")
print(example["target"])
print("\nModel:")
print(answer)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Question:
You are a reliable QA assistant.
Use only the provided context.
If the answer is not supported by the context,
respond exactly with UNANSWERABLE.

Context: The Premier of Victoria is the leader of the political party or coalition with the most seats in the Legislative Assembly. The Premier is the public face of government and, with cabinet, sets the legislative and political agenda. Cabinet consists of representatives elected to either house of parliament. It is responsible for managing areas of government that are not exclusively the Commonwealth's, by the Australian Constitution, such as education, health and law enforcement. The current Premier of Victoria is Daniel Andrews.
Question: What does the secretary need to lead in the Legislative Assembly?

Answer:

Expected:
UNANSWERABLE

Model:
UNANSWERABLE


In [14]:
import re
import string
from collections import Counter

def normalize_text(text):
    text = text.lower()
    text = "".join(
        ch for ch in text
        if ch not in string.punctuation
    )
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def is_correct_prediction(example, prediction):
    prediction = normalize_text(prediction)

    if example["target"] == "UNANSWERABLE":
        return prediction == "unanswerable"

    accepted_answers = [
        normalize_text(answer)
        for answer in example["all_answers"]
    ]
    return prediction in accepted_answers


def token_f1(prediction, ground_truth):
    """SQuAD-style token overlap F1. Gives partial credit to answers that
    are substantively right but not an exact string match (e.g. missing
    an article, or a slightly different span boundary)."""
    pred_tokens = normalize_text(prediction).split()
    gt_tokens = normalize_text(ground_truth).split()

    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return float(pred_tokens == gt_tokens)

    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)


def best_f1(prediction, all_answers):
    return max((token_f1(prediction, ans) for ans in all_answers), default=0.0)


def evaluate_test_set(model, test_data, tokenizer, device):
    total = len(test_data)

    correct = 0
    answerable_total = 0
    answerable_correct = 0
    unanswerable_total = 0
    unanswerable_correct = 0
    answerable_f1_sum = 0.0

    results = []

    for i, example in enumerate(test_data):
        answer = generate_answer(
            model,
            example["prompt"],
            tokenizer,
            device
        )

        expected = example["target"]
        is_correct = is_correct_prediction(example, answer)

        if is_correct:
            correct += 1

        if expected == "UNANSWERABLE":
            unanswerable_total += 1
            if is_correct:
                unanswerable_correct += 1
            f1 = None
        else:
            answerable_total += 1
            if is_correct:
                answerable_correct += 1
            f1 = best_f1(answer, example["all_answers"])
            answerable_f1_sum += f1

        results.append({
            "id": example["id"],
            "expected": expected,
            "model_answer": answer,
            "correct": is_correct,
            "answerable": expected != "UNANSWERABLE",
            "f1": f1,
        })

        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{total}")

    overall_accuracy = correct / total if total > 0 else 0
    answerable_accuracy = answerable_correct / answerable_total if answerable_total else 0
    unanswerable_accuracy = unanswerable_correct / unanswerable_total if unanswerable_total else 0
    mean_answerable_f1 = answerable_f1_sum / answerable_total if answerable_total else 0

    print(f"\nOverall exact-match accuracy: {overall_accuracy:.3f}")
    print(f"Answerable exact-match accuracy: {answerable_accuracy:.3f}")
    print(f"Answerable mean F1 (partial credit): {mean_answerable_f1:.3f}")
    print(f"Unanswerable accuracy (correct abstention rate): {unanswerable_accuracy:.3f}")

    return results


results = evaluate_test_set(test_model, test_data, test_tokenizer, device)


Processed 10/500
Processed 20/500
Processed 30/500
Processed 40/500
Processed 50/500
Processed 60/500
Processed 70/500
Processed 80/500
Processed 90/500
Processed 100/500
Processed 110/500
Processed 120/500
Processed 130/500
Processed 140/500
Processed 150/500
Processed 160/500
Processed 170/500
Processed 180/500
Processed 190/500
Processed 200/500
Processed 210/500
Processed 220/500
Processed 230/500
Processed 240/500
Processed 250/500
Processed 260/500
Processed 270/500
Processed 280/500
Processed 290/500
Processed 300/500
Processed 310/500
Processed 320/500
Processed 330/500
Processed 340/500
Processed 350/500
Processed 360/500
Processed 370/500
Processed 380/500
Processed 390/500
Processed 400/500
Processed 410/500
Processed 420/500
Processed 430/500
Processed 440/500
Processed 450/500
Processed 460/500
Processed 470/500
Processed 480/500
Processed 490/500
Processed 500/500

Overall exact-match accuracy: 0.402
Answerable exact-match accuracy: 0.240
Answerable mean F1 (partial credi

In [15]:
wrong = [
    r for r in results
    if not r["correct"]
]

print(f"Wrong predictions: {len(wrong)}")

for r in wrong[:20]:
    print("\n--------------------")
    print("Expected:", r["expected"])
    print("Model:", r["model_answer"])
    print("Answerable:", r["answerable"])
    if r["f1"] is not None:
        print("F1 (partial credit):", round(r["f1"], 3))


Wrong predictions: 299

--------------------
Expected: UNANSWERABLE
Model: the closer side
Answerable: False

--------------------
Expected: ring of integers of quadratic number fields
Model: algebraic number theory
Answerable: True
F1 (partial credit): 0.25

--------------------
Expected: things that are a matter of custom or expectation
Model: UNANSWERABLE
Answerable: True
F1 (partial credit): 0.0

--------------------
Expected: UNANSWERABLE
Model: Christian Goldbach
Answerable: False

--------------------
Expected: UNANSWERABLE
Model: religious and political freedom
Answerable: False

--------------------
Expected: UNANSWERABLE
Model: California State University at Fresno
Answerable: False

--------------------
Expected: UNANSWERABLE
Model: 1816
Answerable: False

--------------------
Expected: some extra costs
Model: UNANSWERABLE
Answerable: True
F1 (partial credit): 0.0

--------------------
Expected: little Hugos, or those who want Hugo
Model: "Loyal"
Answerable: True
F1 (partial

In [16]:
unanswerable_predictions = sum(
    normalize_text(r["model_answer"]) == "unanswerable"
    for r in results
)

print("Total predictions:", len(results))
print("UNANSWERABLE predictions:", unanswerable_predictions)
print(
    "UNANSWERABLE prediction rate:",
    unanswerable_predictions / len(results)
)

for label, value in [
    ("Correct", sum(r["correct"] for r in results)),
    ("Wrong", sum(not r["correct"] for r in results)),
    ("Model says UNANSWERABLE", unanswerable_predictions),
]:
    print(f"{label}: {value}")

answerable_wrong = [
    r for r in results
    if r["answerable"] and not r["correct"]
]

unanswerable_wrong = [
    r for r in results
    if not r["answerable"] and not r["correct"]
]

answerable_wrong_unanswerable = sum(
    normalize_text(r["model_answer"]) == "unanswerable"
    for r in answerable_wrong
)

unanswerable_wrong_real_answer = sum(
    normalize_text(r["model_answer"]) != "unanswerable"
    for r in unanswerable_wrong
)

print("Answerable wrong:", len(answerable_wrong))
print(
    "Answerable -> incorrectly UNANSWERABLE:",
    answerable_wrong_unanswerable
)

print("\nUnanswerable wrong:", len(unanswerable_wrong))
print(
    "Unanswerable -> incorrectly answered:",
    unanswerable_wrong_real_answer
)


Total predictions: 500
UNANSWERABLE predictions: 267
UNANSWERABLE prediction rate: 0.534
Correct: 201
Wrong: 299
Model says UNANSWERABLE: 267
Answerable wrong: 187
Answerable -> incorrectly UNANSWERABLE: 125

Unanswerable wrong: 112
Unanswerable -> incorrectly answered: 112
